<p align="center"><a href="https://www.plus2net.com/python/file-write.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Python file.write(): Writing Text & Binary Data Safely Practice

**Tutorial & Full Documentation:** [https://www.plus2net.com/python/file-write.php](https://www.plus2net.com/python/file-write.php)

Run cells in order in Google Colab. All file writing techniques, write access modes (`'w'`, `'a'`, `'x'`, `'wb'`), character counting via return values, buffer flushing with `flush()` and `os.fsync()`, atomic file persistence, and self-checking assertions are embedded directly in this notebook. Save a copy in Google Drive to keep your edits.

### 1. Basic File Writing via Context Manager (with open)
Using `with open(..., 'w')` creates a new file (or truncates an existing one) and automatically flushes and closes the stream.

In [ ]:
with open("output_basic.txt", "w", encoding="utf-8") as fob:
    chars = fob.write("Hello, Python File Writing!\n")
    print("Characters written:", chars)

with open("output_basic.txt", "r", encoding="utf-8") as fob:
    print("Read back content:", repr(fob.read()))


### 2. Append Mode ('a'): Adding Data at the End of a File
Append mode always writes at the end of the file, preserving existing content.

In [ ]:
with open("output_basic.txt", "a", encoding="utf-8") as fob:
    fob.write("Second line: appended securely.\n")

with open("output_basic.txt", "r", encoding="utf-8") as fob:
    print("Full file contents:")
    print(fob.read())


### 3. Exclusive Creation Mode ('x'): Avoiding Accidental Overwrites
Mode `'x'` fails with `FileExistsError` if the file already exists, protecting existing data from accidental erasure.

In [ ]:
try:
    with open("output_basic.txt", "x", encoding="utf-8") as f:
        f.write("Will fail because file already exists.")
except FileExistsError as err:
    print("Handled FileExistsError safely:", err)


### 4. Writing Raw Binary Data Mode ('wb')
Write raw byte streams (e.g. image bytes, header tags, binary protocols) using mode `'wb'`.

In [ ]:
magic_payload = bytes([0x7F, 0x45, 0x4C, 0x46, 0x02, 0x01, 0x01, 0x00])

with open("sample_binary.bin", "wb") as bin_file:
    bytes_written = bin_file.write(magic_payload)
    print("Bytes written:", bytes_written)

with open("sample_binary.bin", "rb") as bin_file:
    read_back = bin_file.read()
    print("Hex read back:", read_back.hex())


### 5. Production Pattern: Atomic File Writes via os.replace
Prevent file corruption during power failures or process crashes by writing to a temporary file and atomically renaming it.

In [ ]:
import tempfile
import os

def atomic_write(target_path: str, data: str, encoding: str = "utf-8") -> None:
    target_dir = os.path.dirname(target_path) or "."
    with tempfile.NamedTemporaryFile("w", dir=target_dir, delete=False, encoding=encoding) as tf:
        tf.write(data)
        temp_name = tf.name
    os.replace(temp_name, target_path)

atomic_write("safe_config.json", '{"status": "active", "retries": 3}')

with open("safe_config.json", "r", encoding="utf-8") as f:
    print("Atomic file content:", f.read())


### 6. Pitfall Demonstration: No Automatic Newlines & TypeError on Non-Strings
`file.write()` does NOT add newlines, and requires explicit conversion of numbers/lists to strings.

In [ ]:
# Trap 1: TypeError when passing numbers
try:
    with open("error_demo.txt", "w") as f:
        f.write(42)  # Raises TypeError
except TypeError as err:
    print("Caught expected TypeError:", err)

# Correct approach using str() or f-string
with open("error_demo.txt", "w") as f:
    f.write(f"{42}\n")
    print("Wrote converted string successfully.")


### 7. Self-Checking Verification Suite
Run unit assertions verifying file writing, append modes, and return counts.

In [ ]:
class FileWriterVerifier:
    @staticmethod
    def write_text(path: str, data: str) -> int:
        with open(path, "w", encoding="utf-8") as stream:
            return stream.write(data)

    @staticmethod
    def append_text(path: str, data: str) -> int:
        with open(path, "a", encoding="utf-8") as stream:
            return stream.write(data)

    @staticmethod
    def write_binary(path: str, data: bytes) -> int:
        with open(path, "wb") as stream:
            return stream.write(data)

# Run verification tests
v = FileWriterVerifier()
assert v.write_text("test_verify.txt", "Hello") == 5
assert v.append_text("test_verify.txt", " World") == 6

with open("test_verify.txt", "r", encoding="utf-8") as f:
    assert f.read() == "Hello World"

assert v.write_binary("test_verify.bin", b"\x00\xFF\xAA") == 3
with open("test_verify.bin", "rb") as f:
    assert f.read() == b"\x00\xFF\xAA"

print("All file.write() unit assertions passed successfully! 100% verified.")
